<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Loss: how wrong is the network?

**[Loss: how wrong is the network?](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/)** · Deep Learning and Transformers Explained · Module 3, lesson 1 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a network's scores into probabilities with softmax, measure how wrong they are with the cross-entropy loss, and explain why training uses the loss while reports use accuracy.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account, no graphics card. |
| **You should know** | The forward pass of the ticket network, from [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/). |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M03-L01-loss/loss-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three scores become probabilities

A network gives one **score** per team. Scores can be any number, negative too, so they are not probabilities yet. **Softmax** turns them into probabilities in two steps:

1. Make every score positive with `exp()` (e to the power of the score). A larger score stays larger.
2. Divide each result by their sum, so that the results add up to 1.

Here are three made-up scores for three teams: 2.0, 1.0 and 0.1.

> **Predict.** The first score is twice the second. Will its probability be twice as large, or more than twice? Write your answer, then run the cell.

In [ ]:
scores = torch.tensor([2.0, 1.0, 0.1])
positive = scores.exp()
print("exp(score):   ", positive)
print("sum:          ", positive.sum())
print("probabilities:", positive / positive.sum())
print("softmax:      ", torch.softmax(scores, dim=0))

> **Check.** You should see `exp(score)` 7.3891, 2.7183 and 1.1052, a sum of 11.2125, and the probabilities 0.6590, 0.2424 and 0.0986, twice: once by hand and once with `torch.softmax()`.

The first probability is 2.7 times the second, not 2 times. Softmax compares the **differences** between scores: a score that is 1 higher always gets e ≈ 2.718 times the probability.

## 2. The loss: the surprise at the right answer

The **loss** of one example is −ln p, where p is the probability that the network gave to the **right** team. If the network gave the right team 0.659, the loss is −ln 0.659 = 0.417: a small surprise. If the right team got only 0.0986, the loss is 2.317: a big surprise. This loss is called **cross-entropy**. It is the log loss of the machine learning course, for any number of teams.

PyTorch's `nn.CrossEntropyLoss()` (made as `loss_fn` in the setup) takes the **scores**, not the probabilities, and the **index** of the right team. It does the softmax itself. `scores.unsqueeze(0)` makes the shape (1, 3): a batch of one example.

> **Predict.** What does `loss_fn` give when the right team is the first one (index 0)? Then run the cell.

In [ ]:
import math

right = torch.tensor([0])                     # the index of the right team, for a batch of one
print("by hand:     ", -math.log(0.6590))
print("loss_fn:     ", loss_fn(scores.unsqueeze(0), right))

> **Check.** You should see about `0.417` twice (`tensor(0.4170)` from `loss_fn`).

> **Your turn.** Compute the loss when the right team is the **third** one (index 2). Use `loss_fn`, and put the number in `loss_third` with `.item()`, which turns a one-number tensor into a Python number. Then run the check cell.

In [ ]:
loss_third = loss_fn(scores.unsqueeze(0), torch.tensor([2])).item()
print(loss_third)

In [ ]:
expect('loss_third', loss_third, 2.317, tolerance=0.001)

## 3. The untrained ticket network

Now the real network of [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): 2,864 word counts → 64 hidden neurons with ReLU → 5 scores, one per team. Its weights are random, so it has learned nothing yet. Here is its answer for the first validation ticket, whose right team is **payment** (index 2 in `TEAMS`).

The output tensors end with `grad_fn=…`. That is a note that PyTorch keeps for training; [Backpropagation and autograd](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/) explains it.

> **Predict.** The network knows nothing yet. About what probability will it give each of the five teams? Then run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
scores = model(x_valid[:1])
print("scores:       ", scores)
print("probabilities:", torch.softmax(scores, dim=1))
print("right team:   ", TEAMS[y_valid[0]])
print("loss:         ", loss_fn(scores, y_valid[:1]))

> **Check.** You should see the scores −0.1288, 0.0228, 0.1072, 0.0242, 0.0647, the probabilities 0.1722, 0.2003, 0.2180, 0.2006, 0.2089, the right team `payment` and a loss of `1.5234` (= −ln 0.218).

All five probabilities are close to 0.2: the network guesses about evenly. Now the loss over all 600 validation tickets. `evaluate()` from the course's `ticketnet.py` returns the mean loss and the accuracy.

> **Predict.** If a network gives exactly 0.2 to every team, its loss is −ln 0.2 = ln 5. How close will the untrained network be? Then run the cell.

In [ ]:
valid_loss, valid_acc = evaluate(model, x_valid, y_valid)
print("validation loss:", round(valid_loss, 4), "| ln 5 =", round(math.log(5), 4))
print("validation accuracy:", round(valid_acc, 4))

> **Check.** You should see `validation loss: 1.624 | ln 5 = 1.6094` and `validation accuracy: 0.1817`.

A loss of 1.624 is the loss of even guessing. Training must push it down; a trained network reaches about 0.06.

## 4. Why we train on the loss but report accuracy

Accuracy counts right answers. It changes only when an answer flips from one team to another. The loss changes with every small change in a probability. The next cell makes every ticket's payment score a little lower, by 0.3, 0.2 and 0.1. That is the same as a smaller bias for the payment score.

> **Predict.** Which number moves between the three lines: the loss, the accuracy, or both? Then run the cell.

In [ ]:
scores = model(x_train)
for change in [-0.3, -0.2, -0.1]:
    shifted = scores.clone()
    shifted[:, 2] += change          # every ticket's payment score, a little lower
    loss = loss_fn(shifted, y_train).item()
    accuracy = (shifted.argmax(dim=1) == y_train).float().mean().item()
    print(f"change {change:.2f}   loss {loss:.5f}   accuracy {accuracy:.4f}")

> **Check.** You should see the loss fall from line to line, and the same accuracy, `0.1469`, on all three lines.

The loss says that −0.1 is better than −0.2, and −0.2 better than −0.3. The accuracy cannot tell the three apart. Training needs a number that answers "is this small change a little better or a little worse?" for **every** small change. The loss always answers; accuracy is flat between its jumps. So we train on the loss, and we report accuracy, which people understand.

## 5. Failure case: softmax twice

Tomás adds softmax at the end of the network "to get probabilities", and keeps `nn.CrossEntropyLoss()`. But `loss_fn` applies softmax itself, so the probabilities go through softmax a second time.

> **Predict.** The right team has a probability of 0.659. What loss does `loss_fn` report when it gets the probabilities instead of the scores? Then run the cell.

In [ ]:
probabilities = torch.softmax(torch.tensor([2.0, 1.0, 0.1]), dim=0)
print("scores in:       ", loss_fn(torch.tensor([[2.0, 1.0, 0.1]]), right))
print("probabilities in:", loss_fn(probabilities.unsqueeze(0), right))
perfect = torch.tensor([[1.0, 0.0, 0.0, 0.0, 0.0]])  # 100% on the right team, out of five
print("lowest possible: ", loss_fn(perfect, torch.tensor([0])))

> **Check.** You should see `0.4170`, then `0.8021`, then `0.9048`.

With softmax twice, the loss is wrong. Even a perfect answer gets a loss of 0.905 for five teams, so the loss can never fall below it. **Fix:** let the network return the raw scores, and use `nn.CrossEntropyLoss()` on them. Apply `torch.softmax()` only when you want to read probabilities.

## 6. Check yourself

> **Your turn.** Write `loss_for(scores, right)`: it takes the five scores of one ticket (a tensor of shape (5,)) and the index of the right team, and returns −ln p of the right team, as a Python number. Use `torch.softmax()` and `math.log()`, not `loss_fn`. Then run the check cell, which tries it on the first validation ticket as if its right team were **delivery** (index 0).

In [ ]:
def loss_for(scores, right):
    probabilities = torch.softmax(scores, dim=0)
    return -math.log(probabilities[right].item())

In [ ]:
expect('loss_for(first ticket, delivery)', loss_for(model(x_valid[:1])[0], 0), 1.7591, tolerance=0.001)

## Next

You can now turn scores into probabilities, measure how wrong a network is with the loss, and explain why training needs the loss. The untrained ticket network has a loss of 1.624, the loss of even guessing. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/) has the diagrams, the chart of loss and accuracy and the knowledge checks that count toward your certificate. The next lesson shows how to make the loss smaller: gradient descent.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Gradient descent](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.